# 🏦 End-to-End Pre-Modelling Pipeline
## Bank Customer Churn — EDA → Cleaning → Analysis → Feature Engineering

**Dataset:** Bank Customer Churn (Kaggle)  
**Goal:** Prepare data completely before feeding to any ML model  
**Author:** Ankita | DS Returnship Journey

---

### 📋 Pipeline Overview
```
1. Setup & Data Loading
2. First Look — Shape, Types, Sample
3. Data Quality Check — Missing, Duplicates, Types
4. Univariate Analysis — Each feature individually
5. Bivariate Analysis — Features vs Target
6. Multivariate Analysis — Feature interactions
7. Outlier Detection & Treatment
8. Data Cleaning
9. Feature Engineering — Create new features
10. Encoding Categorical Variables
11. Feature Scaling
12. Feature Selection
13. Final Dataset — Ready for Modelling
```

## 📦 STEP 1 — Setup & Imports

In [ ]:
# ── Core Libraries ──
import numpy as np
import pandas as pd
from scipy import stats

# ── Visualisation ──
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

# ── Preprocessing ──
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.feature_selection import SelectKBest, chi2, f_classif
from sklearn.feature_selection import mutual_info_classif

# ── Settings ──
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## 📥 STEP 2 — Load Data

In [ ]:
# Download from: https://www.kaggle.com/datasets/shivamb/bank-customer-churn-modeling
# Place Churn_Modelling.csv in same folder as this notebook

df = pd.read_csv('Churn_Modelling.csv')

# Keep a clean copy — ALWAYS do this before any changes
df_original = df.copy()

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 🔍 STEP 3 — First Look

In [ ]:
print('=' * 60)
print('SHAPE')
print('=' * 60)
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')

In [ ]:
print('=' * 60)
print('DATA TYPES & NON-NULL COUNTS')
print('=' * 60)
df.info()

In [ ]:
print('=' * 60)
print('STATISTICAL SUMMARY')
print('=' * 60)
df.describe(include='all')

In [ ]:
# Column overview — quick reference
col_summary = pd.DataFrame({
    'dtype':    df.dtypes,
    'nulls':    df.isnull().sum(),
    'null_pct': (df.isnull().sum() / len(df) * 100).round(2),
    'unique':   df.nunique(),
    'sample':   df.iloc[0]
})
print('COLUMN OVERVIEW')
col_summary

## 🧹 STEP 4 — Data Quality Check

In [ ]:
print('=' * 60)
print('MISSING VALUES')
print('=' * 60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percentage': missing_pct})
missing_df = missing_df[missing_df['count'] > 0].sort_values('count', ascending=False)

if len(missing_df) == 0:
    print('✅ No missing values found!')
else:
    print(missing_df)
    # Visualise missing values
    plt.figure(figsize=(10, 4))
    sns.heatmap(df.isnull(), cbar=False, cmap='viridis', yticklabels=False)
    plt.title('Missing Value Heatmap')
    plt.tight_layout()
    plt.show()

In [ ]:
print('=' * 60)
print('DUPLICATE ROWS')
print('=' * 60)
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
if dups > 0:
    print('Sample duplicates:')
    print(df[df.duplicated(keep=False)].head())

In [ ]:
print('=' * 60)
print('TARGET VARIABLE — CLASS BALANCE')
print('=' * 60)
target_counts = df['Exited'].value_counts()
target_pct    = df['Exited'].value_counts(normalize=True) * 100

print(f'Not Churned (0): {target_counts[0]:,} ({target_pct[0]:.1f}%)')
print(f'Churned     (1): {target_counts[1]:,} ({target_pct[1]:.1f}%)')
print(f'\n⚠️ Imbalanced dataset: {target_pct[1]:.1f}% positive class')
print('→ Will need SMOTE or class_weight in modelling phase')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
target_counts.plot(kind='bar', ax=axes[0], color=['steelblue','coral'],
                   edgecolor='white', rot=0)
axes[0].set_title('Churn Distribution (Count)')
axes[0].set_xticklabels(['Not Churned', 'Churned'])
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():,}', (p.get_x()+0.05, p.get_height()+50))

axes[1].pie(target_counts, labels=['Not Churned','Churned'],
            autopct='%1.1f%%', colors=['steelblue','coral'],
            startangle=90)
axes[1].set_title('Churn Distribution (%)')
plt.tight_layout()
plt.show()

## 📊 STEP 5 — Drop Irrelevant Columns

In [ ]:
# RowNumber, CustomerID, Surname — IDs, not features
cols_to_drop = ['RowNumber', 'CustomerId', 'Surname']
df.drop(columns=cols_to_drop, inplace=True)

print(f'Dropped: {cols_to_drop}')
print(f'Remaining columns: {df.shape[1]}')
print(df.columns.tolist())

## 📈 STEP 6 — Univariate Analysis

In [ ]:
# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = df.select_dtypes(include=['object']).columns.tolist()

# Remove target from feature lists
num_features = [c for c in num_cols if c != 'Exited']

print(f'Numerical features: {num_features}')
print(f'Categorical features: {cat_cols}')

In [ ]:
# ── Numerical: Distribution plots ──
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(num_features):
    ax = axes[idx]
    # Histogram + KDE
    ax.hist(df[col].dropna(), bins=30, color='steelblue',
            edgecolor='white', alpha=0.8, density=True)
    df[col].dropna().plot(kind='kde', ax=ax, color='red', linewidth=2)
    ax.axvline(df[col].mean(),   color='orange', linestyle='--',
               linewidth=1.5, label=f'Mean={df[col].mean():.1f}')
    ax.axvline(df[col].median(), color='green',  linestyle='-.',
               linewidth=1.5, label=f'Median={df[col].median():.1f}')
    ax.set_title(f'{col}\nSkew={df[col].skew():.2f}')
    ax.legend(fontsize=7)

# Hide empty subplots
for idx in range(len(num_features), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Numerical Features — Distribution Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Numerical: Box plots for outlier view ──
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(num_features):
    ax = axes[idx]
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    # Calculate and show outlier count
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    outliers = df[(df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)]
    ax.set_title(f'{col}\nOutliers: {len(outliers)}')

for idx in range(len(num_features), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Box Plots — Outlier Detection', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Categorical: Count plots ──
fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 4))
if len(cat_cols) == 1:
    axes = [axes]

for idx, col in enumerate(cat_cols):
    counts = df[col].value_counts()
    axes[idx].bar(counts.index, counts.values,
                  color=sns.color_palette('muted', len(counts)),
                  edgecolor='white')
    axes[idx].set_title(f'{col} Distribution')
    axes[idx].set_xlabel(col)
    for p_idx, v in enumerate(counts.values):
        axes[idx].text(p_idx, v + 50, f'{v:,}\n({v/len(df)*100:.1f}%)',
                      ha='center', fontsize=8)

plt.suptitle('Categorical Features — Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔗 STEP 7 — Bivariate Analysis (Features vs Target)

In [ ]:
# ── Numerical features vs Churn ──
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for idx, col in enumerate(num_features):
    ax = axes[idx]
    churned     = df[df['Exited'] == 1][col]
    not_churned = df[df['Exited'] == 0][col]

    ax.hist(not_churned, bins=25, alpha=0.6, color='steelblue',
            label='Not Churned', density=True, edgecolor='white')
    ax.hist(churned, bins=25, alpha=0.6, color='coral',
            label='Churned', density=True, edgecolor='white')

    # T-test p-value
    t_stat, p_val = stats.ttest_ind(churned.dropna(), not_churned.dropna())
    sig = '✅ Significant' if p_val < 0.05 else '❌ Not Significant'
    ax.set_title(f'{col}\np-value={p_val:.4f} {sig}', fontsize=9)
    ax.legend(fontsize=7)

for idx in range(len(num_features), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Numerical Features vs Churn (T-test p-values)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Churn rate by category ──
fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 5))
if len(cat_cols) == 1:
    axes = [axes]

for idx, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['Exited'].mean().sort_values(ascending=False)
    bars = axes[idx].bar(churn_rate.index, churn_rate.values * 100,
                         color=sns.color_palette('RdYlGn_r', len(churn_rate)),
                         edgecolor='white')
    axes[idx].set_title(f'Churn Rate by {col}')
    axes[idx].set_ylabel('Churn Rate (%)')
    axes[idx].axhline(df['Exited'].mean() * 100, color='blue',
                     linestyle='--', label=f'Overall: {df["Exited"].mean()*100:.1f}%')
    axes[idx].legend()
    for bar in bars:
        axes[idx].text(bar.get_x() + bar.get_width()/2,
                      bar.get_height() + 0.3,
                      f'{bar.get_height():.1f}%', ha='center', fontsize=9)

plt.suptitle('Churn Rate by Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Box plots: Numerical vs Churn ──
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, col in enumerate(num_features):
    sns.boxplot(data=df, x='Exited', y=col,
                palette={0: 'steelblue', 1: 'coral'},
                ax=axes[idx])
    axes[idx].set_title(f'{col} by Churn')
    axes[idx].set_xticklabels(['Not Churned', 'Churned'])

for idx in range(len(num_features), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Feature Distribution by Churn Status', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔥 STEP 8 — Multivariate Analysis

In [ ]:
# ── Correlation Heatmap ──
plt.figure(figsize=(12, 8))
corr_matrix = df[num_features + ['Exited']].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix,
            annot=True, fmt='.2f',
            cmap='RdYlGn', center=0,
            mask=mask, square=True,
            linewidths=0.5,
            annot_kws={'size': 9})
plt.title('Correlation Matrix — All Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Print correlations with target specifically
target_corr = corr_matrix['Exited'].drop('Exited').sort_values(key=abs, ascending=False)
print('\nFeature Correlation with Churn (Exited):')
print(target_corr.to_string())

In [ ]:
# ── Pair plot (top correlated features) ──
top_features = target_corr.abs().nlargest(4).index.tolist() + ['Exited']
sns.pairplot(df[top_features], hue='Exited',
             palette={0: 'steelblue', 1: 'coral'},
             plot_kws={'alpha': 0.4},
             diag_kind='kde')
plt.suptitle('Pair Plot — Top Correlated Features vs Churn',
             y=1.02, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🚨 STEP 9 — Outlier Detection & Treatment

In [ ]:
def detect_outliers_iqr(df, col):
    """Detect outliers using IQR method"""
    q1  = df[col].quantile(0.25)
    q3  = df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return len(outliers), lower, upper

def detect_outliers_zscore(df, col, threshold=3):
    """Detect outliers using Z-score method"""
    z_scores = np.abs(stats.zscore(df[col].dropna()))
    return (z_scores > threshold).sum()

print(f'{"Column":<20} {"IQR Outliers":<15} {"Z-score Outliers":<18} {"Lower":<12} {"Upper"}')
print('-' * 75)

outlier_summary = {}
for col in num_features:
    iqr_count, lower, upper = detect_outliers_iqr(df, col)
    z_count = detect_outliers_zscore(df, col)
    outlier_summary[col] = {'iqr': iqr_count, 'zscore': z_count,
                            'lower': lower, 'upper': upper}
    print(f'{col:<20} {iqr_count:<15} {z_count:<18} {lower:<12.1f} {upper:.1f}')

In [ ]:
# ── Outlier Treatment Options ──
# Option 1: Remove outliers
# Option 2: Cap/Winsorise (replace with fence values) ← recommended for finance
# Option 3: Transform (log, sqrt)
# Option 4: Keep (tree-based models handle outliers well)

df_cleaned = df.copy()

def winsorise(df, col, lower_pct=0.01, upper_pct=0.99):
    """Cap outliers at percentile boundaries"""
    lower = df[col].quantile(lower_pct)
    upper = df[col].quantile(upper_pct)
    original_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'{col}: Winsorised {original_outliers} values → [{lower:.1f}, {upper:.1f}]')
    return df

# Apply winsorisation to columns with significant outliers
# Check outlier_summary above and decide which columns need treatment
cols_to_winsorise = ['CreditScore', 'Age']

for col in cols_to_winsorise:
    df_cleaned = winsorise(df_cleaned, col)

print(f'\n✅ Outlier treatment complete. Shape: {df_cleaned.shape}')

## ⚙️ STEP 10 — Feature Engineering

In [ ]:
# Feature Engineering — Create meaningful new features from existing ones
# This is where your domain knowledge shines!

df_fe = df_cleaned.copy()

# ── 1. Balance per Product ──
# High balance + few products = potential high-value churner
df_fe['BalancePerProduct'] = df_fe['Balance'] / (df_fe['NumOfProducts'] + 1)

# ── 2. Balance to Salary Ratio ──
# Financial health indicator
df_fe['BalanceSalaryRatio'] = df_fe['Balance'] / (df_fe['EstimatedSalary'] + 1)

# ── 3. Is Zero Balance ──
# 0 balance customers are very different — high churn risk!
df_fe['IsZeroBalance'] = (df_fe['Balance'] == 0).astype(int)

# ── 4. Age Group ──
# Age bucketing — younger customers behave differently
df_fe['AgeGroup'] = pd.cut(df_fe['Age'],
                            bins=[0, 30, 40, 50, 60, 100],
                            labels=['<30', '30-40', '40-50', '50-60', '60+'])

# ── 5. Credit Score Category ──
# Domain knowledge: standard credit score buckets
df_fe['CreditScoreCategory'] = pd.cut(df_fe['CreditScore'],
                                       bins=[0, 580, 670, 740, 800, 850, 900],
                                       labels=['Very Poor','Fair','Good',
                                               'Very Good','Exceptional','Elite'])

# ── 6. Tenure per Age ──
# How long have they been a customer relative to their age?
df_fe['TenurePerAge'] = df_fe['Tenure'] / df_fe['Age']

# ── 7. Products × Activity ──
# Active customers with many products = loyal?
df_fe['ProductActivity'] = df_fe['NumOfProducts'] * df_fe['IsActiveMember']

# ── 8. High Value Customer ──
# Domain-defined: high balance + high salary
balance_median = df_fe['Balance'].median()
salary_median  = df_fe['EstimatedSalary'].median()
df_fe['IsHighValue'] = ((df_fe['Balance'] > balance_median) &
                         (df_fe['EstimatedSalary'] > salary_median)).astype(int)

# ── 9. Complaint Proxy ──
# Low tenure + inactive + no CC = potential dissatisfied customer
df_fe['DissatisfiedProxy'] = ((df_fe['Tenure'] < 3) &
                               (df_fe['IsActiveMember'] == 0) &
                               (df_fe['HasCrCard'] == 0)).astype(int)

new_features = ['BalancePerProduct', 'BalanceSalaryRatio', 'IsZeroBalance',
                'AgeGroup', 'CreditScoreCategory', 'TenurePerAge',
                'ProductActivity', 'IsHighValue', 'DissatisfiedProxy']

print(f'✅ Created {len(new_features)} new features!')
print(f'Features: {new_features}')
print(f'\nDataset shape: {df_fe.shape}')
df_fe[new_features].head()

In [ ]:
# ── Validate new features — do they actually predict churn? ──
print('New Feature Churn Rates:')
print('─' * 40)

# Binary features — churn rate when feature = 1
for col in ['IsZeroBalance', 'IsHighValue', 'DissatisfiedProxy', 'ProductActivity']:
    if df_fe[col].nunique() == 2:
        rate_0 = df_fe[df_fe[col]==0]['Exited'].mean()
        rate_1 = df_fe[df_fe[col]==1]['Exited'].mean()
        print(f'{col}: 0→{rate_0:.1%}, 1→{rate_1:.1%} | Lift: {rate_1/rate_0:.2f}x')

print('\nAgeGroup Churn Rates:')
print(df_fe.groupby('AgeGroup', observed=True)['Exited'].mean().apply(lambda x: f'{x:.1%}'))

print('\nCreditScoreCategory Churn Rates:')
print(df_fe.groupby('CreditScoreCategory', observed=True)['Exited'].mean().apply(lambda x: f'{x:.1%}'))

## 🔤 STEP 11 — Encoding Categorical Variables

In [ ]:
df_encoded = df_fe.copy()

# ── Label Encoding — Binary or Ordinal categories ──
le = LabelEncoder()

# Gender — binary
df_encoded['Gender_encoded'] = le.fit_transform(df_encoded['Gender'])
print(f"Gender mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# CreditScoreCategory — ordinal (order matters)
credit_order = {'Very Poor': 0, 'Fair': 1, 'Good': 2,
                'Very Good': 3, 'Exceptional': 4, 'Elite': 5}
df_encoded['CreditScore_ord'] = df_encoded['CreditScoreCategory'].map(credit_order)

# AgeGroup — ordinal
age_order = {'<30': 0, '30-40': 1, '40-50': 2, '50-60': 3, '60+': 4}
df_encoded['AgeGroup_ord'] = df_encoded['AgeGroup'].map(age_order)

# ── One-Hot Encoding — Nominal categories (no order) ──
# Geography — no inherent order
geo_dummies = pd.get_dummies(df_encoded['Geography'],
                             prefix='Geo',
                             drop_first=True)  # drop_first avoids multicollinearity
df_encoded = pd.concat([df_encoded, geo_dummies], axis=1)
print(f"\nGeography OHE columns created: {geo_dummies.columns.tolist()}")

# ── Drop original categorical columns ──
cols_to_drop_enc = ['Gender', 'Geography', 'AgeGroup', 'CreditScoreCategory']
df_encoded.drop(columns=cols_to_drop_enc, inplace=True)

print(f'\n✅ Encoding complete. Shape: {df_encoded.shape}')

## ⚖️ STEP 12 — Feature Scaling

In [ ]:
from sklearn.model_selection import train_test_split

# ── Split first — ALWAYS scale AFTER splitting ──
# Scaling before split = data leakage!

X = df_encoded.drop('Exited', axis=1)
y = df_encoded['Exited']

# Remove any remaining NaN
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.2%}')
print(f'Test  churn rate: {y_test.mean():.2%}')
print('✅ Stratified split — class balance preserved!')

In [ ]:
# ── Which scaler to use? ──
# StandardScaler — when data is roughly normal (mean=0, std=1)
# MinMaxScaler   — when you need values in [0,1] range
# RobustScaler   — when data has many outliers (uses IQR)

# For tree-based models (XGBoost, RF) — scaling NOT needed!
# For linear models, KNN, SVM, neural nets — scaling IS needed!

# Columns to scale — only continuous numerical, not binary flags
scale_cols = ['CreditScore', 'Age', 'Tenure', 'Balance',
              'EstimatedSalary', 'BalancePerProduct',
              'BalanceSalaryRatio', 'TenurePerAge']

# Filter to only columns that exist
scale_cols = [c for c in scale_cols if c in X_train.columns]

# ── StandardScaler ──
scaler_std = StandardScaler()
X_train_std = X_train.copy()
X_test_std  = X_test.copy()

# Fit on TRAIN only, transform both
X_train_std[scale_cols] = scaler_std.fit_transform(X_train[scale_cols])
X_test_std[scale_cols]  = scaler_std.transform(X_test[scale_cols])

# ── RobustScaler (better with outliers) ──
scaler_rob = RobustScaler()
X_train_rob = X_train.copy()
X_test_rob  = X_test.copy()

X_train_rob[scale_cols] = scaler_rob.fit_transform(X_train[scale_cols])
X_test_rob[scale_cols]  = scaler_rob.transform(X_test[scale_cols])

print(f'✅ Scaling complete on {len(scale_cols)} columns')
print(f'Scaled: {scale_cols}')
print(f'\nBefore scaling — Balance mean: {X_train["Balance"].mean():.0f}')
print(f'After StandardScaler — Balance mean: {X_train_std["Balance"].mean():.4f}')

## 🎯 STEP 13 — Feature Selection

In [ ]:
# ── Method 1: Correlation with target ──
correlations = X_train.corrwith(y_train).abs().sort_values(ascending=False)
print('Feature Correlation with Target:')
print(correlations.head(10).to_string())

plt.figure(figsize=(10, 6))
correlations.head(15).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Correlation with Churn (absolute)')
plt.xlabel('Absolute Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# ── Method 2: Mutual Information (captures non-linear relationships) ──
mi_scores = mutual_info_classif(X_train.fillna(0), y_train, random_state=42)
mi_df = pd.Series(mi_scores, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(10, 6))
mi_df.head(15).plot(kind='barh', color='mediumseagreen', edgecolor='white')
plt.title('Mutual Information Scores — Feature Importance')
plt.xlabel('Mutual Information Score')
plt.tight_layout()
plt.show()

print('Top 10 features by Mutual Information:')
print(mi_df.head(10).to_string())

In [ ]:
# ── Method 3: SelectKBest ──
selector = SelectKBest(score_func=f_classif, k=10)
selector.fit(X_train.fillna(0), y_train)

selected_features = X_train.columns[selector.get_support()].tolist()
print(f'Top 10 features by SelectKBest (ANOVA F-test):')
print(selected_features)

# ── Drop low importance features ──
# Keep features with MI > 0.01 (tune this threshold)
important_features = mi_df[mi_df > 0.01].index.tolist()
print(f'\n✅ Keeping {len(important_features)} features with MI > 0.01')

X_train_final = X_train_std[important_features]
X_test_final  = X_test_std[important_features]

## ✅ STEP 14 — Final Summary & Export

In [ ]:
print('=' * 60)
print('PRE-MODELLING PIPELINE SUMMARY')
print('=' * 60)
print(f'Original dataset:     {df_original.shape}')
print(f'After cleaning:       {df_cleaned.shape}')
print(f'After feature eng:    {df_fe.shape}')
print(f'After encoding:       {df_encoded.shape}')
print(f'Final X_train:        {X_train_final.shape}')
print(f'Final X_test:         {X_test_final.shape}')
print()
print(f'Target distribution (train):')
print(f'  Not Churned: {(y_train==0).sum():,} ({(y_train==0).mean():.1%})')
print(f'  Churned:     {(y_train==1).sum():,} ({(y_train==1).mean():.1%})')
print()
print(f'Final features ({len(important_features)}):')
for f in important_features:
    print(f'  - {f}')

In [ ]:
# ── Save processed datasets ──
X_train_final.to_csv('X_train_processed.csv', index=False)
X_test_final.to_csv('X_test_processed.csv', index=False)
y_train.to_csv('y_train.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

# Save the scaler for later use in API
import joblib
joblib.dump(scaler_std, 'scaler_standard.pkl')
joblib.dump(scaler_rob, 'scaler_robust.pkl')

print('✅ All processed files saved!')
print('Files: X_train_processed.csv, X_test_processed.csv')
print('       y_train.csv, y_test.csv')
print('       scaler_standard.pkl, scaler_robust.pkl')
print()
print('🚀 Ready for modelling! Next step: Train XGBoost + LightGBM')

---
## 📋 Quick Reference — What Each Step Does

| Step | What | Why |
|------|------|-----|
| First Look | Shape, types, sample | Understand what you have |
| Quality Check | Missing, duplicates, imbalance | Find problems early |
| Drop Columns | Remove IDs, irrelevant cols | Reduce noise |
| Univariate | Each feature alone | Understand distributions |
| Bivariate | Features vs target | Find predictive features |
| Multivariate | Feature interactions | Find multicollinearity |
| Outlier Treatment | Winsorise or remove | Prevent model distortion |
| Feature Engineering | Create new features | Add domain knowledge |
| Encoding | Convert categories to numbers | ML needs numbers |
| Scaling | Normalise ranges | Equal treatment of features |
| Feature Selection | Keep important, drop noise | Reduce overfitting |

## ⚠️ Golden Rules
- **Always keep a copy** of original data before any changes
- **Always split before scaling** — prevents data leakage  
- **Fit scaler on train only** — transform both train and test
- **Document every decision** — why you dropped/kept each feature
- **Validate new features** — check if they actually correlate with target